# Sistema de Recomendação por Imagens (Visual Similarity Recommender)

Este notebook implementa um sistema de recomendação que sugere produtos
**visualmente parecidos** entre si — não pela ficha técnica (preço, marca, modelo),
mas pela sua **aparência** (formato, cor, textura), usando uma rede neural
convolucional pré-treinada como extratora de características (feature extractor).

**Pipeline geral:**
1. Carregar uma rede CNN pré-treinada (ResNet50, treinada no ImageNet) sem a camada de classificação final.
2. Passar cada imagem do catálogo pela rede e guardar o vetor de características (embedding) gerado.
3. Para uma imagem de consulta (query), gerar seu embedding e comparar com todos os embeddings do catálogo usando **similaridade de cosseno**.
4. Retornar os *top-N* produtos mais parecidos visualmente.

> Baseado na estrutura conceitual do notebook de referência:
> https://colab.research.google.com/github/sparsh-ai/rec-tutorials/blob/master/_notebooks/2021-04-27-image-similarity-recommendations.ipynb

**Como usar este notebook:**
- Rode no Google Colab (recomendado, para ter GPU gratuita e fácil acesso a bibliotecas).
- Ajuste a variável `DATASET_DIR` para apontar para uma pasta com suas imagens, organizadas em subpastas por classe (ex: `dataset/relogio/`, `dataset/camiseta/`, `dataset/tenis/`).
- Não tem um dataset em mãos? A seção 1.1 mostra como baixar um pequeno dataset de exemplo público direto no Colab.


## 1. Configuração do ambiente

In [ ]:
# Bibliotecas necessárias (no Colab a maioria já vem pré-instalada)
!pip install -q tensorflow scikit-learn matplotlib pillow


In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.preprocessing import image as keras_image
from tensorflow.keras.models import Model

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors

print("TensorFlow:", tf.__version__)
print("GPU disponível:", tf.config.list_physical_devices('GPU'))


### 1.1 (Opcional) Baixando um dataset de exemplo

Se você ainda não tem um catálogo próprio de imagens, pode usar um subconjunto
público de imagens de produtos de moda para testar o pipeline (ex: o dataset
"Fashion Product Images (Small)" do Kaggle, ou qualquer pasta de imagens
organizada por classe que você já tenha).

Estrutura de pastas esperada:

```
dataset/
├── relogio/
│   ├── img_001.jpg
│   ├── img_002.jpg
├── camiseta/
│   ├── img_001.jpg
├── tenis/
│   ├── img_001.jpg
├── bicicleta/
│   ├── img_001.jpg
```


In [ ]:
DATASET_DIR = "dataset"  # ajuste para o caminho do seu dataset

# Descobre todas as imagens dentro das subpastas (classes)
image_paths = glob.glob(os.path.join(DATASET_DIR, "*", "*.jpg")) + \
              glob.glob(os.path.join(DATASET_DIR, "*", "*.png")) + \
              glob.glob(os.path.join(DATASET_DIR, "*", "*.jpeg"))

print(f"Total de imagens encontradas: {len(image_paths)}")
image_paths[:5]


## 2. Carregando o extrator de características (Feature Extractor)

Usamos a ResNet50 pré-treinada no ImageNet, removendo a camada final de
classificação (`include_top=False`) e aplicando Global Average Pooling.
O resultado é um vetor de 2048 dimensões que representa a "aparência" da imagem —
cor, textura, formas — de forma muito mais rica do que qualquer descrição textual.


In [ ]:
base_model = ResNet50(weights="imagenet", include_top=False, pooling="avg")
# pooling='avg' já aplica GlobalAveragePooling2D, retornando um vetor (2048,) por imagem

base_model.trainable = False  # não vamos re-treinar, só usar como extrator
base_model.summary()


## 3. Função para extrair o embedding de uma imagem

In [ ]:
IMG_SIZE = (224, 224)  # tamanho esperado pela ResNet50

def get_embedding(img_path, model=base_model):
    """Carrega uma imagem, pré-processa e retorna seu vetor de embedding."""
    img = keras_image.load_img(img_path, target_size=IMG_SIZE)
    x = keras_image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)
    embedding = model.predict(x, verbose=0)
    return embedding.flatten()


## 4. Construindo o índice de embeddings do catálogo

Aqui passamos todas as imagens do catálogo pela rede uma única vez e guardamos
os embeddings em memória (em produção, isso seria feito em lote/offline e
armazenado em um banco vetorial como FAISS, Pinecone, Milvus, etc.).


In [ ]:
embeddings = []
valid_paths = []

for path in image_paths:
    try:
        emb = get_embedding(path)
        embeddings.append(emb)
        valid_paths.append(path)
    except Exception as e:
        print(f"Erro ao processar {path}: {e}")

embeddings = np.array(embeddings)
print("Shape da matriz de embeddings:", embeddings.shape)


## 5. Motor de recomendação por similaridade

Usamos `NearestNeighbors` do scikit-learn com métrica de cosseno, que é o
padrão para comparar embeddings de imagens (mede o ângulo entre os vetores,
não a magnitude — mais robusto a variações de brilho/escala).


In [ ]:
knn = NearestNeighbors(n_neighbors=6, metric="cosine")  # 6 pq o 1º resultado é a própria imagem
knn.fit(embeddings)

def recommend_similar(query_path, top_n=5):
    """Retorna os top_n produtos mais parecidos visualmente com a imagem de consulta."""
    query_embedding = get_embedding(query_path).reshape(1, -1)
    distances, indices = knn.kneighbors(query_embedding, n_neighbors=top_n + 1)

    # remove a própria imagem de consulta do resultado, se ela estiver no catálogo
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        candidate_path = valid_paths[idx]
        if candidate_path == query_path:
            continue
        similarity_score = 1 - dist  # cosine distance -> similaridade
        results.append((candidate_path, similarity_score))

    return results[:top_n]


## 6. Visualizando as recomendações

In [ ]:
def show_recommendations(query_path, top_n=5):
    results = recommend_similar(query_path, top_n=top_n)

    plt.figure(figsize=(15, 4))
    plt.subplot(1, top_n + 1, 1)
    plt.imshow(Image.open(query_path))
    plt.title("Consulta (Query)")
    plt.axis("off")

    for i, (path, score) in enumerate(results):
        plt.subplot(1, top_n + 1, i + 2)
        plt.imshow(Image.open(path))
        plt.title(f"Similaridade: {score:.2f}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

# Exemplo de uso — troque pelo caminho de uma imagem real do seu dataset
if valid_paths:
    show_recommendations(valid_paths[0], top_n=5)


## 7. Avaliando os resultados

Algumas perguntas úteis para validar qualitativamente o sistema:

- As imagens recomendadas pertencem à mesma classe da consulta (ex: "tênis" recomenda outros "tênis")?
- Dentro da mesma classe, as recomendações realmente têm cor/formato/textura parecidos?
- O sistema recomenda produtos de classes diferentes só porque têm aparência parecida (ex: um relógio redondo recomendando um prato redondo)? Isso é esperado, já que o sistema ignora completamente metadados textuais.

Uma avaliação quantitativa mais rigorosa usaria métricas como Precision@K,
comparando as recomendações com rótulos de categoria conhecidos.


In [ ]:
# Avaliação simples: quantas das top-N recomendações pertencem à mesma classe (pasta) da consulta?
def classe_da_imagem(path):
    return os.path.basename(os.path.dirname(path))

def avaliar_precisao_classe(query_path, top_n=5):
    resultados = recommend_similar(query_path, top_n=top_n)
    classe_query = classe_da_imagem(query_path)
    acertos = sum(1 for path, _ in resultados if classe_da_imagem(path) == classe_query)
    return acertos / top_n

if valid_paths:
    precisoes = [avaliar_precisao_classe(p, top_n=5) for p in valid_paths]
    print(f"Precisão média (mesma classe) @5: {np.mean(precisoes):.2%}")


## 8. Próximos passos (evolução do projeto)

- **Escala**: para catálogos grandes (milhões de produtos), trocar `NearestNeighbors`
  por um índice vetorial otimizado como **FAISS** (Facebook AI Similarity Search) ou **Annoy**.
- **Fine-tuning**: treinar a última(s) camada(s) da CNN com imagens do domínio específico
  (ex: moda, autopeças) para embeddings ainda mais discriminativos.
- **API de produção**: expor a função `recommend_similar` via uma API REST (Flask/FastAPI),
  recebendo a imagem do usuário e devolvendo os IDs dos produtos recomendados.
- **Combinação híbrida**: unir similaridade visual com filtros textuais (categoria, faixa de
  preço) para recomendações mais relevantes no contexto de um e-commerce real.
